In [1]:
import jax
import netket as nk

import numpy as np
import jax.numpy as jnp

# from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule
from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule 

In [2]:
import jax
import jax.numpy as jnp
from functools import partial

from netket.jax import COOArray
from netket.utils.types import Array

@jax.jit
def hamming_distance(x, y):
    """
    Args:
        x, y: 1D JAX arrays of 0/1 values with the same shape.

    Returns:
      d : Hamming distance (scalar)
    """
    return jnp.sum(x != y)
  
@partial(jax.jit, static_argnames=['k'])
def select_changes(x: Array, y: Array, k: int = 4):
    diff = x - y  
    size = k // 2
    k_destroy = jnp.argwhere(diff == 1, size=size, fill_value=-1).reshape(-1)
    l_create = jnp.argwhere(diff == -1, size=size, fill_value=-1).reshape(-1)
    return jnp.flip(k_destroy), jnp.flip(l_create)


@jax.jit
def jw_sign_fast(x, k_destroy, l_create):
    """
    Fast and correct Jordan–Wigner sign that matches the original implementation.
    """

    prefix = jnp.cumsum(x) - x # number of ones to the left
    parity_destroy = jnp.sum(prefix[k_destroy])

    xd = x.at[k_destroy].set(0) # apply destruction BEFORE computing create parity
    prefix_d = jnp.cumsum(xd) - xd # prefix after destruction

    parity_create = jnp.sum(prefix_d[l_create])

    total_parity = (parity_destroy + parity_create) & 1 # total parity
    return 1 - 2 * total_parity # (-1)^parity


@jax.jit
@partial(jnp.vectorize, signature="(n)->()", excluded=(0, 2, 3, 4))
def _get_mel_offdiagonal(
    x: Array,
    y: Array,
    index_array: Array | COOArray | None,
    create_array: Array | None,
    weight_array: Array,
):
    r"""
    Get the matrix element between two states `x` and `y` for two-body operators
    of the form 
    
    .. math::
        c^\dagger_{i} c^\dagger_{j} c_{k} c_{l}
        
    Args:
        x: Array
            Initial state (1D array of 0/1 values).
        y: Array
            Final state (1D array of 0/1 values).
        index_array: Array | COOArray | None
            Precomputed index array for the two-body operator.
        create_array: Array | None
            Precomputed creation array for the two-body operator.
        weight_array: Array
            Precomputed weight array for the two-body operator.
            
    Returns:
        mel: Array
            The matrix element connecting `x` to `y`.
    """
    def compute(k_destroy, l_create, ind):
        creates = create_array[ind] # shape (n_max, 4)
        
        # idx = jnp.all(creates == l_create[..., None, :], axis=-1)
        mask = jnp.all(creates == l_create[..., None, :], axis=-1)
        idx = jnp.sum(jnp.arange(creates.shape[-2]) * mask, axis=-1)
        
        sgn = jw_sign_fast(x, k_destroy, l_create)
        return sgn * weight_array[ind, idx]

    def case_k4():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 4,
        corresponding to two-body operator transitions between different sites, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{k} c_{l} with i,j,k,l all different.
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0]
        """
        k_destroy, l_create = select_changes(x, y, k=4)
        ind = index_array[tuple(k_destroy)]
        return compute(k_destroy, l_create, ind)

    def case_k2():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 2,
        corresponding to two-body operator transitions that involve the destruction
        and creation of a particle in the same site, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{j} c_{k}  or  c^\dagger_{i} c^\dagger_{i} c_{k} c_{l}
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]
        """
        k_destroy_, l_create_ = select_changes(x, y, k=2) # identifies the two sites that are different
        same_sites = jnp.where((x & y), size=3, fill_value=-1)[0] # all sites where a particle could have been both destroyed and created
        
        def f_one_site(i):
            r"""
            Computes the matrix element contribution for a single same-site transition.
            Args:
                i: Index of the site where a particle is both destroyed and created.
            Returns:
                Matrix element contribution for this specific same-site transition.
            """
            k_destroy = jnp.sort(jnp.array([k_destroy_[0], i]), descending=True)
            l_create = jnp.sort(jnp.array([l_create_[0], i]), descending=True)
            ind = index_array[tuple(k_destroy)]
            return compute(k_destroy, l_create, ind)

        mels = jax.vmap(f_one_site)(same_sites) # vectorize over all i in same_sites
        valid_mask = same_sites != -1 # mask out padding (-1 entries)
        return jnp.sum(mels * valid_mask)

    d = hamming_distance(x, y)
    return jnp.where(d == 4, case_k4(), jnp.where(d == 2, case_k2(), 0.0))


@jax.jit
@partial(jnp.vectorize, signature="(n),(n)->()", excluded=(0, 1, 4, 5, 6))
def _get_mel_mixed_offdiag(
    x_down: Array,
    x_up: Array,
    y_down: Array,
    y_up: Array,
    index_array: Array,
    create_array: Array,
    weight_array: Array,
):
    """
    Compute matrix element for mixed_offdiag (cross-sector) two-body operators.
    
    Handles transitions where one sector has a hop and the other has same-site.
    """
    
    d_down = hamming_distance(x_down, y_down)
    d_up = hamming_distance(x_up, y_up)
    
    def case_hop_in_down():
        # Hop in down, same-site in up
        k_destroy_down, l_create_down = select_changes(x_down, y_down, k=2)
        same_sites_up = jnp.where((x_up & y_up), size=3, fill_value=-1)[0]
        
        def f_one_site(j_up):
            ind = index_array[k_destroy_down[0], j_up]
            creates = create_array[ind]
            target = jnp.array([l_create_down[0], j_up])
            idx = jnp.argmax(jnp.all(creates == target, axis=1))
            
            weight = weight_array[ind, idx]
            sgn = jw_sign_fast(x_down, k_destroy_down, l_create_down)
            return sgn * weight
        
        mels = jax.vmap(f_one_site)(same_sites_up)
        valid_mask = same_sites_up != -1
        return jnp.sum(mels * valid_mask)
    
    def case_hop_in_up():
        # Hop in up, same-site in down
        k_destroy_up, l_create_up = select_changes(x_up, y_up, k=2)
        same_sites_down = jnp.where((x_down & y_down), size=3, fill_value=-1)[0]
        
        def f_one_site(j_down):
            ind = index_array[j_down, k_destroy_up[0]]
            creates = create_array[ind]
            target = jnp.array([j_down, l_create_up[0]])
            idx = jnp.argmax(jnp.all(creates == target, axis=1))
            
            weight = weight_array[ind, idx]
            sgn = jw_sign_fast(x_up, k_destroy_up, l_create_up)
            return sgn * weight
        
        mels = jax.vmap(f_one_site)(same_sites_down)
        valid_mask = same_sites_down != -1
        return jnp.sum(mels * valid_mask)
    
    # Select based on which sector has the hop
    return jnp.where(
        (d_down == 2) & (d_up == 0),
        case_hop_in_down(),
        jnp.where(
            (d_down == 0) & (d_up == 2),
            case_hop_in_up(),
            0.0
        )
    )
    

In [3]:
mol, mo_coeff, mf = PCMolecule.molecule(cid=62714)
molecule = PCMolecule(mol=mol, mo_coeff=mo_coeff)

H = molecule.hamiltonian.to_jax_operator()
hi = molecule.hilbert_space

using 2d
Hartree-Fock energy: -7.767362135748573
E(CCSD) = -7.784454825913955  E_corr = -0.01709269016538233
CCSD energy: -7.784454825913955


/Users/lucagravina/venvs/neuralimportancesampling/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [4]:
all_states = hi.all_states()
print("all_states_size =", all_states.shape)

print("n_fermions =", hi.n_fermions)
print("n_orbitals =", hi.n_orbitals)
print("n_connected_states =", H.max_conn_size)

n_max = hi.n_orbitals * 2

all_states_size = (225, 12)
n_fermions = 4
n_orbitals = 6
n_connected_states = 91


In [5]:
_operator_data = H._operator_data
print("_operator_data keys =", _operator_data.keys())

_operator_data keys = dict_keys(['diag', 'offdiag', 'mixed_diag', 'mixed_offdiag'])


In [6]:
_operator_data['offdiag'].keys()

dict_keys([(2, (np.int32(0), np.int32(1))), (4, (np.int32(0), np.int32(1)))])

In [7]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            (k,s): v for (k,s), v in inner_dict.items() 
            if filter_func(k,s)
        }
    return result

In [8]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import unpack_spin_sectors, get_conn_padded_pnc_spin

x = jnp.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0])

_operator_data_filtered = filter_keys(_operator_data, lambda k,s: k == 4)
_operator_data_filtered['diag'] = _operator_data_filtered['mixed_diag'] = _operator_data_filtered['mixed_offdiag'] = {}

n_fermions_per_spin = hi.n_fermions_per_spin
n_spin_subsectors = len(n_fermions_per_spin)

xp, mels = get_conn_padded_pnc_spin(_operator_data_filtered, x, n_fermions_per_spin)

y = xp[3]

is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))
    y_index = y_index
    print("matrix element connecting x to y =", mels[y_index], "sum = ", jnp.sum(mels[y_index]))

xs = unpack_spin_sectors(x, n_spin_subsectors)
ys = unpack_spin_sectors(y, n_spin_subsectors)

print("\nxs =", xs)
print("ys =", ys)

Is y in xp? True
matrix element connecting x to y = [0.00712389] sum =  0.007123885146157282

xs = (Array([1, 1, 0, 0, 0, 0], dtype=int64), Array([0, 0, 1, 1, 0, 0], dtype=int64))
ys = (Array([0, 1, 0, 0, 0, 1], dtype=int64), Array([0, 0, 1, 1, 0, 0], dtype=int64))


In [9]:
(k, sectors) = (4, (0,1))
index_array, create_array, weight_array = _operator_data_filtered['offdiag'][(k, sectors)]

mel = 0.0
for i in sectors:
    mel_ = _get_mel_offdiagonal(xs[i], ys[i], index_array, create_array, weight_array)
    print("mel for sector", i, "=", mel_)
    mel += mel_
mel

mel for sector 0 = 0.007123885146157282
mel for sector 1 = 0.0


Array(0.00712389, dtype=float64)

In [10]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import unpack_spin_sectors, get_conn_padded_pnc_spin

x = jnp.array([1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0])

_operator_data_filtered = filter_keys(_operator_data, lambda k,s: k == 4)
_operator_data_filtered['diag'] = _operator_data_filtered['mixed_diag'] = _operator_data_filtered['offdiag'] = {}

n_fermions_per_spin = hi.n_fermions_per_spin
n_spin_subsectors = len(n_fermions_per_spin)

xp, mels = get_conn_padded_pnc_spin(_operator_data_filtered, x, n_fermions_per_spin)

y = xp[2]

is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))
    y_index = y_index
    print("matrix element connecting x to y =", mels[y_index], "sum = ", jnp.sum(mels[y_index]))

xs = unpack_spin_sectors(x, n_spin_subsectors)
ys = unpack_spin_sectors(y, n_spin_subsectors)

print("\nxs =", xs)
print("ys =", ys)

Is y in xp? True
matrix element connecting x to y = [-0.0210682   0.04857828] sum =  0.027510083349398366

xs = (Array([1, 1, 0, 0, 0, 0], dtype=int64), Array([0, 0, 1, 1, 0, 0], dtype=int64))
ys = (Array([1, 1, 0, 0, 0, 0], dtype=int64), Array([0, 0, 0, 1, 0, 1], dtype=int64))


In [11]:
(k, sectors) = (4, ((1, 0),))
index_array, create_array, weight_array = _operator_data['mixed_offdiag'][(k, sectors)]

mel = _get_mel_mixed_offdiag(
    xs[0], xs[1], ys[0], ys[1],
    index_array, create_array, weight_array
)

mel

Array(0.02751008, dtype=float64)

In [12]:
len(unpack_spin_sectors(xp))

2

In [17]:
y = xp[2:5]
ys = unpack_spin_sectors(y, n_spin_subsectors)
ys[1]

Array([[0, 0, 0, 1, 0, 1],
       [0, 0, 1, 1, 0, 0],
       [0, 0, 1, 1, 0, 0]], dtype=int64)